## Instalation

In [ ]:
!pip install transformers datasets accelerate huggingface_hub pyyaml

## Login in hugginFace

In [ ]:
from huggingface_hub import login

login()

## Global Settings

In [ ]:
import os

CONFIG = {
    "seed": 42,

    "teacher_name": "meta-llama/Llama-2-7b-chat-hf",
    "student_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    "epochs": 3,
    "batch_size": 2,
    "lr": 2e-5,
    "temperature": 4.0,

    "save_steps": 50,

    "raw_data_path": "/content/datasetTrain.json",
    "distilled_data_path": "data/train.jsonl",
    "processed_path": "data/processed.json",

    "output_dir": "outputs",
    "log_file": "outputs/logs.jsonl"
}

os.makedirs("data", exist_ok=True)
os.makedirs(CONFIG["output_dir"], exist_ok=True)
os.makedirs(CONFIG["output_dir"] + "/checkpoints", exist_ok=True)

## Test Data

In [ ]:
import json
import os

os.makedirs("data", exist_ok=True)

raw_data = [
    {
        "instruction": "Explain what machine learning is.",
        "context": "",
        "response": "Machine learning is a field of artificial intelligence that allows systems to learn from data."
    },
    {
        "instruction": "What is overfitting?",
        "context": "",
        "response": "Overfitting occurs when a model learns the training data too well and fails to generalize."
    },
    {
        "instruction": "Define neural networks.",
        "context": "",
        "response": "Neural networks are models inspired by the human brain composed of layers of interconnected nodes."
    },
    {
        "instruction": "Explain supervised learning.",
        "context": "",
        "response": "Supervised learning is when a model is trained using labeled data."
    },
    {
        "instruction": "What is a dataset?",
        "context": "",
        "response": "A dataset is a collection of data used to train or evaluate a model."
    }
]

with open("data/raw_train.json", "w") as f:
    json.dump(raw_data, f, indent=2)

print("raw_train.json created")

raw_train.json created


In [ ]:
import json
import torch
import os

os.makedirs("data", exist_ok=True)

VOCAB_SIZE = 100
MAX_LEN = 128

def generate_fake_logits():
    return torch.randn(MAX_LEN, VOCAB_SIZE).tolist()

with open("/content/datasetTrain.json", "r") as f:
    data = json.load(f)


if isinstance(data, dict) and "data" in data:
    data = data["data"]

with open("data/train.jsonl", "w") as out:

    for i, item in enumerate(data):


        if not isinstance(item, dict):
            print(f" item invalid {i}: {item}")
            continue

        prompt = f"""### Instruction:
{item['instruction']}

### Context:
{item.get('context', '')}

### Response:
"""

        out.write(json.dumps({
            "id": i,
            "prompt": prompt,
            "teacher_output": item["response"],
            "logits": generate_fake_logits()
        }) + "\n")

print("train.jsonl created")

train.jsonl created


## Seed

In [ ]:
import torch
import random
import numpy as np

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

## Process DataSet

In [ ]:
import json

with open(CONFIG["raw_data_path"], "r") as f:
    data = json.load(f)


if isinstance(data, dict) and "data" in data:
    data = data["data"]

processed = []

for i, item in enumerate(data):


    if not isinstance(item, dict):
        print(f"Item isn't dict in index {i}: {item}")
        continue

    if "instruction" not in item or "response" not in item:
        print(f"Item invalid index {i}: {item}")
        continue

    instruction = item["instruction"]
    context = item.get("context", "")
    response = item["response"]

    prompt = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
"""

    processed.append({
        "id": i,
        "prompt": prompt,
        "response": response
    })

# Save Dataset
with open(CONFIG["processed_path"], "w") as f:
    json.dump(processed, f, indent=2)

print("Dataset generated")
print("Total samples:", len(processed))

# Debug
print("\nEjemplo procesado:")
print(processed[0])

Dataset generated
Total samples: 20

Ejemplo procesado:
{'id': 0, 'prompt': '### Instruction:\nIdentify the main purpose of the document.\n\n### Context:\n\n\n### Response:\n', 'response': 'The main purpose of the document is to approve the International Telecommunication Convention, along with its Regulations, signed on December 10, 1932, by the Colombian delegates at the International Telecommunication Conference in Madrid.'}


## Load Models


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

teacher_tokenizer = AutoTokenizer.from_pretrained(CONFIG["teacher_name"])
student_tokenizer = AutoTokenizer.from_pretrained(CONFIG["student_name"])

if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token

if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(CONFIG["teacher_name"]).to(device)
student = AutoModelForCausalLM.from_pretrained(CONFIG["student_name"]).to(device)

teacher.eval()

Device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

## Generate JsonL

In [ ]:
import json

MAX_LEN = 128

with open(CONFIG["processed_path"], "r") as f:
    data = json.load(f)

with open(CONFIG["distilled_data_path"], "w") as out:

    for sample in data:

        prompt = sample["prompt"]
        response = sample["response"]


        full_input = prompt + response

        inputs = teacher_tokenizer(
            full_input,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN
        ).to(device)

        with torch.no_grad():
            outputs = teacher(**inputs)

        logits = torch.clamp(outputs.logits.squeeze(0), -10, 10).cpu().tolist()

        teacher_output = teacher_tokenizer.decode(
            torch.argmax(outputs.logits, dim=-1)[0]
        )

        out.write(json.dumps({
            "id": sample["id"],
            "prompt": prompt,
            "teacher_output": response,
            "logits": logits
        }) + "\n")

print("JSONL generated")

JSONL generated


## DataSet

In [ ]:
from torch.utils.data import Dataset
import json
import torch

MAX_LEN = 128

class DistillationDataset(Dataset):
    def __init__(self, path, tokenizer):
        self.data = [json.loads(line) for line in open(path)]
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        full_text = sample["prompt"] + sample["teacher_output"]

        tokens = self.tokenizer(
            full_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=128
        )

        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)

        prompt_tokens = self.tokenizer(
            sample["prompt"],
            return_tensors="pt",
            truncation=True,
            max_length=128
        )

        prompt_len = prompt_tokens["input_ids"].size(1)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "teacher_logits": torch.tensor(sample["logits"], dtype=torch.float32),
            "prompt_len": prompt_len
        }

## Training

In [ ]:
from torch.utils.data import DataLoader
import torch.nn.functional as F
import time
import os
import json

dataset = DistillationDataset(CONFIG["distilled_data_path"], student_tokenizer)

loader = DataLoader(
    dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)

optimizer = torch.optim.AdamW(student.parameters(), lr=CONFIG["lr"])

checkpoint_path = CONFIG["output_dir"] + "/checkpoints/last.pt"

start_epoch = 0
global_step = 0


if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)

    student.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    start_epoch = checkpoint["epoch"]
    global_step = checkpoint["step"]

    print("Checkpoint loaded")


alpha = 0.7

for epoch in range(start_epoch, CONFIG["epochs"]):

    for batch in loader:

        global_step += 1

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        teacher_logits = batch["teacher_logits"].to(device)
        prompt_len = batch["prompt_len"]

        outputs = student(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        student_logits = outputs.logits


        #  SHIFT


        shift_student = student_logits[:, :-1, :]
        shift_teacher = teacher_logits[:, 1:, :]
        shift_labels  = input_ids[:, 1:]


        # SOFT LOSS (DISTILLATION)


        T = CONFIG["temperature"]

        teacher_probs = F.softmax(shift_teacher / T, dim=-1)
        student_log_probs = F.log_softmax(shift_student / T, dim=-1)

        soft_loss = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="none"
        )


        # Mask


        mask = torch.zeros_like(input_ids, dtype=torch.float32)
        pad_token_id = student_tokenizer.pad_token_id

        for i in range(input_ids.size(0)):
            start = prompt_len[i].item()

            for j in range(start, input_ids.size(1)):
                if input_ids[i, j] != pad_token_id:
                    mask[i, j] = 1

        mask = mask[:, 1:]  # shift
        mask = mask.to(device)
        mask = mask.unsqueeze(-1).expand_as(soft_loss)

        soft_loss = (soft_loss * mask).sum() / (mask.sum() + 1e-8)
        soft_loss = soft_loss * (T ** 2)


        # HARD LOSS


        hard_loss = F.cross_entropy(
            shift_student.reshape(-1, shift_student.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=pad_token_id
        )


        # LOSS


        loss = alpha * soft_loss + (1 - alpha) * hard_loss



        if torch.isnan(loss) or torch.isinf(loss):
            print("NaN detected, skipping batch")
            continue

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()

        # checkpoint
        if global_step % CONFIG["save_steps"] == 0:
            torch.save({
                "model": student.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
                "step": global_step
            }, checkpoint_path)

        # log
        log = {
            "epoch": epoch,
            "step": global_step,
            "loss": loss.item(),
            "soft_loss": soft_loss.item(),
            "hard_loss": hard_loss.item(),
            "time": time.time()
        }

        with open(CONFIG["log_file"], "a") as f:
            f.write(json.dumps(log) + "\n")

    print(f"Epoch {epoch} | loss: {loss.item():.4f} | soft: {soft_loss.item():.4f} | hard: {hard_loss.item():.4f}")

Epoch 0 | loss: 0.0611 | soft: 0.0001 | hard: 0.2031
Epoch 1 | loss: 0.0489 | soft: 0.0001 | hard: 0.1631
Epoch 2 | loss: 0.0511 | soft: 0.0001 | hard: 0.1699


## Save Model

In [ ]:
student.save_pretrained(CONFIG["output_dir"] + "/final_model")
student_tokenizer.save_pretrained(CONFIG["output_dir"] + "/final_model")

print("Final model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved


## Load Distilli Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_path = CONFIG["output_dir"] + "/final_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path).to(device)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

print("Model Load Succesfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model Load Succesfully


## Ask Function

In [ ]:
def ask_model(instruction, context=""):

    prompt = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=None,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)


    if "### Response:" in full_text:
        response = full_text.split("### Response:")[-1]
    else:
        response = full_text

    return response.strip()

## Test

In [ ]:
print(ask_model("Tell me about EAFIT"))


EAFIT is a private university located in Medellín, Colombia. The university was founded in 1963 and has since become one of the leading institutions of higher education in Colombia. EAFIT offers a wide range of undergraduate and graduate programs in fields such as business, law, engineering, and sciences. The university has a reputation for offering high-quality education and is known for its innovative approach to teaching. EAFIT is committed to providing its students with a comprehensive education that will enable them to succeed in their professional careers. Its location in the heart of Colombia makes it an attractive choice for students who want to study in a vibrant and cosmopolitan city. EAFIT is a member of the Latin American Association of Universities and the Council of Latin American Universities. It has collaborated with several international universities and organizations, including the University of North Carolina at Chapel Hill and the University of Copenhagen. EAFIT has

## Compresss Model

In [ ]:
!zip -r final_model.zip outputs/final_model

  adding: outputs/final_model/ (stored 0%)
  adding: outputs/final_model/tokenizer_config.json (deflated 46%)
  adding: outputs/final_model/chat_template.jinja (deflated 60%)
  adding: outputs/final_model/generation_config.json (deflated 29%)
  adding: outputs/final_model/model.safetensors (deflated 21%)
  adding: outputs/final_model/config.json (deflated 49%)
  adding: outputs/final_model/tokenizer.json (deflated 85%)


In [ ]:
import shutil

shutil.make_archive("final_model", 'zip', "outputs/final_model")

'/content/final_model.zip'

In [ ]:
from google.colab import files
files.download("final_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>